# I. 독립변수 16개(t-test + logistic regression) QSVM-KTA(커널-타깃 정렬) 버전

## 1. 라이브러리 설치

In [ ]:
# Colab에서 실행할 때만 아래 주석을 풀고, 프로젝트 폴더 경로를 맞춰 주세요.
# (로컬 Jupyter / VS Code에서는 이 노트북이 있는 폴더에서 그대로 실행)
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/<프로젝트 폴더>

## 2. 데이터 불러오기 및 탐색 함수

In [ ]:
import numpy as np
import pandas as pd

from config import (
    SEEDS, QUBIT_RANGE, SELECTION_METHODS, ENCODINGS, REDUCTIONS,
)
from src.common import (
    load_raw, run_eda, preprocess, build_df16, split16, iterate_splits,
)
from config import KTA_GAMMAS, REPS_KTA, N_KTA
from src.qsvm import kta_sweep
from src.metrics import save_sweep, summarize_results

NOTEBOOK = "ind16_QSVM_KTA"



# 데이터: data/FILE2_중공업.xls (config.DATA_PATH)
df = load_raw()
df.tail(5)

## 3. 전체 탐색 실행

In [ ]:
# 탐색 함수와 출력 내용은 src/common.py 의 run_eda
x, dates, tables, summary = run_eda(df)

## 4. 전처리

### 기존 전처리

In [ ]:
# 전처리 클래스와 파이프라인은 src/common.py 의 preprocess
prep = preprocess(df)

### 독립변수 16개 추출

In [ ]:
# 148개 전처리 결과에서 16개 변수만 뽑아 df16 만들기 (src/common.py 의 build_df16)
df16 = build_df16(prep).df16
df16.head()

#### 데이터 분할

In [ ]:
# 기준 seed(config.SEED) 분할 확인: 148개 버전과 같은 분할인지 내부에서 검증
# (실험 셀에서는 seed마다 prepare_split 으로 다시 나누고 변수도 다시 고른다)
sp = split16(df16, prep)

## 6. QSVM-KTA (커널-타깃 정렬)

### 구성

기존 QSVM은 각도 스케일 γ를 **검증셋 AUC**로 골랐다. 여기서는 **KTA(Kernel-Target Alignment, 커널-타깃 정렬)** 로 양자 커널의 설정을 고른다.

| 개념 | 설명 |
|---|---|
| KTA | 커널 행렬이 "같은 클래스끼리 비슷하고 다른 클래스끼리 다른" 정답 구조(y yᵀ)와 얼마나 닮았는지를 나타내는 값 (중심화 KTA, −1 ~ 1, 클수록 좋음) |
| 장점 | 훈련셋의 라벨만으로 계산하고 SVM을 학습하지 않아도 되므로, 검증셋을 설정 선택에 쓰지 않는다 |
| 한계 | KTA가 높다고 분류 성능이 반드시 높은 것은 아니다. 최종 판단은 검증·테스트 성능으로 한다 |

| 모델 | 구성 |
|---|---|
| A. QSVM-KTA (특징 맵 선택) | γ(`config.KTA_GAMMAS`)와 reps(1~3, iqp에서만)의 조합 중 훈련 부분표본의 KTA가 가장 높은 것을 선택 |
| B. QSVM-KTA (+특징별 가중치) | A에서 출발해, 입력 i에 곱하는 가중치 wᵢ를 KTA가 오르는 방향으로 학습 (하나씩 키우거나 줄여 보고 오르면 채택, 2회 탐색) |

**양자 인코딩** (`config.ENCODINGS`, 양자 모델에만 적용)

| 인코딩 | 양자 상태 | 입력 수 d (n큐비트) |
|---|---|---|
| basic | 기존 방식: QSVM은 IQP 특징 맵, VQC는 RY 각도 | n |
| angle | 큐비트마다 RY(θ), 얽힘 없음 | n |
| iqp | H → Z·ZZ 위상 반복 (인접 큐비트 얽힘), QSVM·VQC 공통 | n |
| dense_angle | 큐비트 하나에 변수 2개: RY(θ₁) 후 RZ(θ₂) | 2n (n ≤ 8) |
| amplitude | 입력 벡터를 정규화해 상태 진폭으로 사용 | 2ⁿ (n ≤ 4) |

**입력 차원 맞추기** (`config.REDUCTIONS`): 선택된 변수 16개를 d개로 줄인다.

| 방법 | 처리 |
|---|---|
| pca | 표준화 → PCA d개 성분 → 성분 표준화 (기존 방식) |
| select | 변수 선택 순위 상위 d개 → 표준화 |

- d = 16이면 줄이지 않고 표준화만 한다. 각도형 인코딩은 θ = π·tanh(z/2)로 바꾼다.
- 쓸 수 없는 큐비트 수(예: amplitude 5큐비트 이상)는 자동으로 건너뛴다.
- basic + pca에서 n = 8은 기존 8큐비트(PCA 8개), n = 16은 기존 16큐비트 입력과 같다.
- 고전 모델은 PCA 없이 선택된 변수 16개를 그대로 쓴다.

- KTA 계산에는 훈련셋에서 클래스 비율을 맞춰 뽑은 부분표본(`config.N_KTA` = 300개)을 쓴다. 부분표본도 seed마다 달라진다. 최종 SVM 학습은 훈련셋 전체로 한다.
- 분류기는 SVC(precomputed kernel)이고, **C만** 검증셋 ROC-AUC로 고른다. 테스트셋은 최종 평가에만 쓴다.
- 가중치 학습은 부분표본에 과적합될 수 있으므로, 검증셋 KTA와 검증·테스트 성능으로 확인한다.


### 실험 설계: 변수 선택 × 인코딩 × 큐비트 수 × seed

`config.py`의 목록을 모두 조합해 반복한다.

| 설정 | 내용 | 기본값 |
|---|---|---|
| `SELECTION_METHODS` | 변수 선택 방법 (seed마다 **훈련셋에서만** 선택) | `["fixed16"]` |
| `ENCODINGS` | 양자 인코딩 (basic / angle / iqp / dense_angle / amplitude) | `["basic"]` |
| `REDUCTIONS` | 큐비트보다 입력이 많을 때 줄이는 방법 (pca / select) | `["pca"]` |
| `QUBIT_RANGE` | 큐비트 수 | 1 ~ 16 |
| `SEEDS` | 반복 seed (분할 · 전처리 · 선택 · 모델 난수가 모두 바뀜) | 5개 |

seed마다 다시 하는 것: train/valid/test 분할(60/20/20, 층화) → 전처리 fit → 변수 후보(148개 + 업력) → 변수 선택 → 인코딩 fit → 모델 학습.

- 결과는 조합 하나가 끝날 때마다 `results/results.csv`에 저장되므로, 중간에 멈춰도 그때까지의 결과는 남는다.
- 마지막 셀에서 **평균 ± 표준오차**(95% 신뢰구간 포함)로 요약한다.
- 실행 시간은 조합 수에 비례한다. 먼저 목록을 줄여서 돌려 본다 (예: `SEEDS = [42]`, `QUBIT_RANGE = [4, 8, 16]`).
- `fixed16`은 전체 데이터로 고른 기존 목록이라 테스트 정보가 섞여 있다. 다른 방법(훈련셋에서만 선택)과의 차이가 선택 편향의 크기다.


### QSVM-KTA (변수 선택 × 인코딩 × 1~16큐비트 × seed)


In [ ]:
# KTA 함수와 실험 순서는 src/qsvm.py (kta_select_feature_map, kta_learn_weights)
for data in iterate_splits(df, SELECTION_METHODS, SEEDS):
    for encoding in ENCODINGS:
        for reduction in REDUCTIONS:
            summary_a, summary_b, runs = kta_sweep(
                QUBIT_RANGE, data.X, data.Y,
                gammas=KTA_GAMMAS, reps_list=REPS_KTA, size=N_KTA,
                encoding=encoding, reduction=reduction, seed=data.seed,
            )

            tags = dict(
                seed=data.seed, selection=data.selection,
                encoding=f"{encoding}/{reduction}",
            )

            save_sweep(
                NOTEBOOK, "QSVM-KTA (특징 맵 선택)", runs, key="feature_map", **tags,
                params_of=lambda info: {"gamma": info["gamma"], "reps": info["reps"], "C": info["C"]},
            )
            save_sweep(
                NOTEBOOK, "QSVM-KTA (+특징별 가중치)", runs, key="weights", **tags,
                params_of=lambda info: {
                    "gamma": info["gamma"], "reps": info["reps"], "C": info["C"],
                    "weights": np.round(info["weights"], 4).tolist(),
                },
            )

## 모델 성능 비교

In [ ]:
# ============================================================
# 요약: 평균 ± 표준오차 (테스트셋)
# ============================================================

summary_table, summary_full = summarize_results(notebook=NOTEBOOK)

print(f"[{NOTEBOOK}: 테스트 성능, 평균 ± 표준오차]")
print(summary_table.to_string())

# 지표별 n, mean, std, se, 95% 신뢰구간 전체 표
summary_full.round(4)

In [ ]:
# 참고: 다른 노트북 결과 (results/results.csv, 테스트셋, 평균 ± 표준오차)
reference_table, _ = summarize_results(exclude_notebook=NOTEBOOK)

print("[참고: 다른 노트북 결과]")
print(reference_table.to_string())

> **참고:** 아래 보고서는 8큐비트와 16큐비트만, seed 하나로 실험하던 **이전 실험(리팩터링 전)** 기록이다.
> 현재 코드는 변수 선택 방법 × 인코딩 × 1~16큐비트 × 여러 seed로 실험한다.


## 7. 연구보고서

### 7.1 목적
t-test와 로지스틱 회귀로 고른 독립변수 16개를 입력으로, QSVM의 설정을 KTA(커널-타깃 정렬)로 고르거나 학습하면 성능이 달라지는지 확인한다. 8큐비트(PCA 8개)와 16큐비트(변수 16개를 큐비트 16개에 직접 입력)를 각각 본다.

### 7.2 데이터와 입력
- 3,140개 기업(정상/폐업 1:1), 훈련/검증/테스트 = 1,884 / 628 / 628 (148개 버전과 동일한 분할임을 확인)
- 입력 변수 16개: 영업이익률, 재료비/매출액, 감가상각률, 금융비용대총부채비율, 금융비용대총비용비율, EBITDA/총차입금, 차입금의존도, 장단기대여금/총자산, 법인세부담률, 순운전자본대총자본비율, 유동자산구성비율, 유보액대총자산비율, 재고자산대유동자산비율, 매입채무회전기간, 자본금회전율(회), 업력
- 8큐비트: 16개 변수 → 표준화 → PCA 8개 → 표준화 → θ = π·tanh(z/2). PCA 누적 설명 분산 0.7462
- 16큐비트: PCA 없이 변수 1개를 큐비트 1개에 입력. 따라서 특징별 가중치를 변수별 가중치로 읽을 수 있다.

### 7.3 방법
- 양자 특징 맵: IQP/ZZ 계열, 인접 큐비트 ZZ 얽힘, 충실도 커널, 분류기는 SVC(precomputed)
- 모델 A (특징 맵 선택): γ와 reps(1~3) 조합 중 훈련 부분표본(8큐비트 400개, 16큐비트 300개)의 KTA가 최대인 조합 선택
- 모델 B (+특징별 가중치): A에서 출발해 특징별 각도 가중치를 KTA가 오르는 방향으로 2회 탐색
- C는 두 모델 모두 검증셋 ROC-AUC로 선택, 테스트셋은 최종 평가에만 사용

### 7.4 결과
**선택된 설정**

| 모델 | γ | reps | 부분표본 KTA (선택 → 가중치 학습 후) | 검증셋 KTA (학습 전 → 후) | C | 검증 AUC |
|---|---:|---:|---|---|---:|---:|
| 8큐비트 A | 0.15 | 2 | 0.1789 | - | 10 | 0.8534 |
| 8큐비트 B | 0.15 | 2 | 0.1789 → 0.2995 | 0.1504 → 0.2513 | 100 | 0.8511 |
| 16큐비트 A | 0.10 | 2 | 0.2943 | - | 10 | 0.8807 |
| 16큐비트 B | 0.10 | 2 | 0.2943 → 0.4383 | 0.2337 → 0.3442 | 10 | 0.8799 |

**테스트 성능**

| 모델 | Accuracy | Precision | Recall | F1 | ROC-AUC | Avg. precision | Log loss |
|---|---:|---:|---:|---:|---:|---:|---:|
| QSVM-KTA 8큐비트 A | 0.8424 | 0.8248 | 0.8694 | 0.8465 | 0.8980 | 0.8721 | 0.3978 |
| QSVM-KTA 8큐비트 B | 0.8408 | 0.8242 | 0.8662 | 0.8447 | 0.9001 | 0.8843 | 0.3961 |
| QSVM-KTA 16큐비트 A | 0.8583 | 0.8399 | 0.8854 | 0.8620 | 0.9268 | 0.9217 | 0.3419 |
| QSVM-KTA 16큐비트 B | 0.8503 | 0.8374 | 0.8694 | 0.8531 | 0.9238 | 0.9172 | 0.3520 |

(Balanced accuracy는 테스트셋이 314/314라서 Accuracy와 같다.)

**테스트 혼동행렬 (행: 실제, 열: 예측, 순서: 정상/폐업)**

| 모델 | 정상→정상 | 정상→폐업 | 폐업→정상 | 폐업→폐업 |
|---|---:|---:|---:|---:|
| 8큐비트 A | 256 | 58 | 41 | 273 |
| 8큐비트 B | 256 | 58 | 42 | 272 |
| 16큐비트 A | 261 | 53 | 36 | 278 |
| 16큐비트 B | 261 | 53 | 41 | 273 |

**기존 QSVM(검증셋 AUC로 γ 선택) 대비 차이 (KTA − 기존)**

| 모델 | ΔAccuracy | ΔROC-AUC | ΔLog loss |
|---|---:|---:|---:|
| 8큐비트 A | +0.0048 | +0.0007 | -0.0005 |
| 8큐비트 B | +0.0032 | +0.0028 | -0.0022 |
| 16큐비트 A | 0.0000 | 0.0000 | 0.0000 |
| 16큐비트 B | -0.0080 | -0.0030 | +0.0101 |

기존 값: 8큐비트 0.8376 / 0.8973 / 0.3983, 16큐비트 0.8583 / 0.9268 / 0.3419 (Accuracy / AUC / Log loss). 16큐비트 A는 기존과 수치가 완전히 같아서, 두 방법이 같은 설정을 골랐을 가능성이 높다.

**참고 모델 (이전 실험, 테스트)**

| 모델 | Accuracy | ROC-AUC | Log loss |
|---|---:|---:|---:|
| XGBoost(16개 변수) | 0.8615 | 0.9453 | 0.3044 |
| XGBoost(PCA 8개) | 0.8232 | 0.8946 | 0.4052 |
| VQC(8큐비트) | 0.6990 | 0.7791 | 0.5619 |
| VQC(8큐비트+drc) | 0.7532 | 0.8313 | 0.5128 |

### 7.5 해석
- KTA로 설정을 골라도 기존 QSVM과 성능이 사실상 같다. 차이는 Accuracy ±0.008, AUC ±0.003 이내이고, 테스트 AUC 표준오차(약 0.015)보다 작다.
- 특징별 가중치 학습(B)은 KTA를 가장 크게 올린 경우다. 부분표본 KTA는 0.1789 → 0.2995(8큐비트), 0.2943 → 0.4383(16큐비트)로 올랐고, 검증셋 KTA도 0.1504 → 0.2513, 0.2337 → 0.3442로 함께 올랐다. 그런데 검증 AUC와 테스트 성능은 오르지 않았다. 커널이 라벨 구조와 더 맞춰져도 SVM의 분류 성능은 그대로였다.
- 16큐비트 B의 가중치는 변수별로 읽을 수 있다. 감가상각률(2.0), 금융비용대총비용비율(1.5), 유보액대총자산비율(1.5)이 커졌고, 영업이익률·차입금의존도·EBITDA/총차입금·업력 등 9개 변수는 0.375로 줄었다. 좌표 탐색의 거친 배율이므로 통계적 변수 중요도로 해석하면 안 된다.
- 16큐비트가 8큐비트보다 높다(AUC 0.898~0.900 → 0.924~0.927). 16큐비트는 PCA 없이 변수를 그대로 쓰므로, PCA 8개로 압축하며 잃은 정보(누적 설명 분산 0.7462)가 되살아난 효과로 보인다.
- 16큐비트 KTA-QSVM(AUC 0.924~0.927)은 XGBoost(AUC 0.9453)보다 낮고, VQC 계열(0.78~0.83)보다 높다. Accuracy는 XGBoost(0.8615)와 비슷한 0.850~0.858이다.
- reps=3에서 γ=0.2로 키우면 KTA가 급락한다(0.1470). 특징 맵이 깊고 각도가 클수록 커널이 단위행렬에 가까워지는 현상으로 보인다.

### 7.6 한계
- 데이터 분할 1개, 시드 1개 결과이고, 모두 시뮬레이터(상태벡터) 기반이다.
- 설정 선택용 KTA는 훈련 부분표본(400개/300개)으로 계산했다. 가중치 학습은 부분표본에 과적합될 수 있다.
- 16큐비트 가중치 학습은 약 220초, 최종 학습은 모델당 약 95초가 걸렸다.
- 업력(K1)이 포함된 변수 집합이라, 업력 산출 시점에 따른 데이터 누수 가능성은 기존 16개 버전과 동일하게 남아 있다.